# CSE558 Midterm: Do large hosts drag London Airbnb ratings down?

Inside Airbnb, London snapshot of **17 September 2026** (CC BY 4.0). This notebook reproduces
every number, table and figure in the midterm report, top to bottom from a clean kernel:

1. get the data and check its checksum, 2. build the data dictionary, 3. clean, 4. split into
exploration and confirmation sets **before** any exploration, 5. encode and check the size floor,
6. explore (exploration split only), 7. load the pre-registered hypotheses, 8. run the
confirmatory tests (confirmation split only).

Run from the repository root with `jupyter nbconvert --to notebook --execute --inplace notebooks/midterm_analysis.ipynb`.

In [1]:
import json, platform, sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy, statsmodels, matplotlib
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import prep, dictionary, figures, stats_tests as st

RAW = ROOT / "data" / "raw" / "listings.csv.gz"
TABLES, FIGS, DOCS = ROOT / "outputs" / "tables", ROOT / "outputs" / "figures", ROOT / "docs"
for d in (TABLES, FIGS, DOCS):
    d.mkdir(parents=True, exist_ok=True)

def _jsonable(o):
    if isinstance(o, dict):
        return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if hasattr(o, "item"):          # numpy scalars
        return o.item()
    return o

def save_json(obj, name):
    path = TABLES / name
    path.write_text(json.dumps(_jsonable(obj), indent=2, default=str))
    return path

print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scipy", scipy.__version__, "| statsmodels", statsmodels.__version__, "| matplotlib", matplotlib.__version__)
print("split seed:", prep.SPLIT_SEED, "| exploration share of hosts:", prep.EXPLORATION_SHARE)

python 3.13.15 | numpy 2.5.4 | pandas 3.1.0 | scipy 1.18.1 | statsmodels 0.15.0 | matplotlib 3.11.2
split seed: 558 | exploration share of hosts: 0.3


## 1. Get the data

The file is downloaded straight from Inside Airbnb (not a Kaggle re-upload) and its SHA-256 is
checked, so a TA re-running this gets exactly the snapshot analysed in the report.

In [2]:
prep.ensure_raw(RAW)
raw = prep.load_raw(RAW)
audit = prep.raw_audit(raw)
save_json(audit, "raw_audit.json")
print(f"{audit['rows']:,} rows x {audit['columns']} columns = {audit['cells']:,} cells")
print(f"hosts: {audit['hosts']:,} | empty columns: {audit['n_empty_columns']}")
print(f"price missing: {audit['price_missing']:,} ({audit['price_missing_pct']:.1f}%), of which "
      f"{audit['price_missing_from_previous_scrape']:,} carried over from a previous scrape")
print(f"zero availability: {audit['price_missing_zero_availability_pct']:.1f}% of priceless vs "
      f"{audit['price_present_zero_availability_pct']:.1f}% of priced listings")
print(f"rating missing: {audit['rating_missing']:,} ({audit['rating_missing_pct']:.1f}%) | rating < 1: {audit['rating_below_1']}")
print(f"exact duplicates: {audit['exact_duplicates']} | same host+name+coordinates: {audit['dup_same_host_name_coords']}")
print(f"largest London portfolio: {audit['max_london_listings_per_host']} listings "
      f"(Airbnb-wide count up to {audit['max_airbnb_wide_listings_per_host']:,})")

91,954 rows x 90 columns = 8,275,860 cells
hosts: 51,348 | empty columns: 15
price missing: 32,049 (34.9%), of which 29,724 carried over from a previous scrape
zero availability: 87.3% of priceless vs 0.5% of priced listings
rating missing: 20,694 (22.5%) | rating < 1: 3
exact duplicates: 0 | same host+name+coordinates: 173
largest London portfolio: 501 listings (Airbnb-wide count up to 5,777)


In [3]:
# Currency check: the raw quotes say which currency the '$' prices are in.
currency = raw["price_quote_raw"].dropna().map(lambda s: json.loads(s).get("quote", {}).get("currency"))
print(currency.value_counts(dropna=False))

price_quote_raw
GBP    60028
NaN     2701
Name: count, dtype: int64


## 2. Data dictionary (every attribute: type, unit, range, missingness)

In [4]:
clean, prep_log = prep.clean_deterministic(raw)
dd = dictionary.build(raw, clean)
dd.to_csv(DOCS / "data_dictionary.csv", index=False)
# Markdown version for the repository
lines = ["| attribute | origin | type | kind | unit | range | missing (%) | meaning |", "|---|---|---|---|---|---|---|---|"]
for r in dd.itertuples():
    lines.append(f"| `{r.attribute}` | {r.origin} | {r.type} | {r.kind} | {r.unit} | {r.range} | {r.missing:,} ({r.missing_pct}) | {r.meaning} |")
(DOCS / "data_dictionary.md").write_text("# Data dictionary: Inside Airbnb London, 17 September 2026\n\n"
    "Computed from the raw file by `src/dictionary.py`. Type follows the course scheme "
    "(nominal, discrete, continuous); kind gives the storage form.\n\n" + "\n".join(lines) + "\n")
print(dd["type"].value_counts().to_dict(), "| raw attributes:", int((dd.origin == "raw").sum()),
      "| derived:", int((dd.origin == "derived").sum()))
dd.head(12)

{'nominal': 40, 'discrete': 40, 'continuous': 21} | raw attributes: 90 | derived: 11


,attribute,origin,type,kind,unit,range,missing,missing_pct,meaning
0,id,raw,nominal,identifier,-,"91,954 unique",0,0.0,Airbnb listing ID
1,listing_url,raw,nominal,URL,-,"91,954 unique",0,0.0,listing page on airbnb.com
2,scrape_id,raw,nominal,identifier,-,1 unique,0,0.0,Inside Airbnb scrape run (constant)
3,last_scraped,raw,discrete,date,day,2026-09-18 to 2026-09-23,0,0.0,day the listing was scraped
4,source,raw,nominal,category,-,1 levels: previous scrape,59643,64.9,'previous scrape' = carried over from an earli...
5,name,raw,nominal,free text,-,"89,060 distinct; 1 to 254 characters",0,0.0,listing title written by the host
6,description,raw,nominal,free text (HTML),-,"80,818 distinct; 1 to 1000 characters",701,0.8,listing description written by the host
7,neighborhood_overview,raw,nominal,free text,-,-,91954,100.0,empty in this snapshot
8,picture_url,raw,nominal,URL,-,"90,177 unique",5,0.0,cover photo
9,host_id,raw,nominal,identifier,-,"51,348 unique",0,0.0,Airbnb host ID


## 3. Deterministic cleaning

Row-wise fixes that learn nothing from the data, so they can run before the split. Each step
logs how many rows (or columns) it touched.

In [5]:
log = pd.DataFrame(prep_log)
log.to_csv(TABLES / "prep_log.csv", index=False)
print(f"rows: {len(raw):,} -> {len(clean):,} (dropped {len(raw) - len(clean):,} duplicate units)")
pd.set_option("display.max_colwidth", 120)
log

rows: 91,954 -> 91,775 (dropped 179 duplicate units)


,step,action,touched,reason
0,15 columns are 100% empty,dropped,15,"Airbnb stopped exposing these fields (e.g. host_since, license, instant_bookable); nothing to impute from"
1,price_quote_* repeat price; scrape metadata constant,dropped,7,price_quote_price_per_night equals price for 99.8% of priced listings
2,price stored as text '$267.50',parsed to a number in GBP,59905,format fix; Airbnb's own quotes give the currency as GBP
3,bathrooms only as text ('1.5 shared baths'),parsed to count + shared/private flags,91846,47 text levels carry two facts: how many and whether shared
4,review score averages below 1,set to missing and flagged,7,"a 1-5 star average cannot be below 1; these are recording errors, not bad listings"
5,"minimum_nights up to 1,125",capped at 365 and flagged,24,"1,125 is Airbnb's form maximum, a placeholder for 'long lets only', not a real rule"
6,HTML tags in descriptions,stripped,47950,"tags such as <br /> are markup, not words"
7,same unit listed twice,"dropped extra copies, kept the one with most reviews",179,"173 same-host copies and 6 cross-account copies; 95 of all copies belong to hosts with 10+ listings, the group P1 st..."


## 4. Exploration / confirmation split (before any exploration)

Whole hosts are assigned to one side, because a host's listings share one operator and would
otherwise leak information across the split. 30% of hosts go to exploration (all EDA and every
choice of test settings), 70% to confirmation (only the pre-registered tests). Seed 558.

In [6]:
explore, confirm = prep.split_by_host(clean)
assert not set(explore.host_id) & set(confirm.host_id)
split_rows = []
for name, part in [("exploration", explore), ("confirmation", confirm), ("total", clean)]:
    row = {"split": name, "listings": len(part), "hosts": part.host_id.nunique()}
    for cls in ["single (1)", "small (2-9)", "large (10+)"]:
        row[f"listings {cls}"] = int((part.host_class == cls).sum())
    split_rows.append(row)
split_summary = pd.DataFrame(split_rows)
split_summary.to_csv(TABLES / "split_summary.csv", index=False)
assignment = pd.concat([explore[["id", "host_id"]].assign(split="exploration"),
                        confirm[["id", "host_id"]].assign(split="confirmation")]).sort_values("id")
assignment.to_csv(TABLES / "split_assignment.csv.gz", index=False)
split_summary

,split,listings,hosts,listings single (1),listings small (2-9),listings large (10+)
0,exploration,27586,15404,12372,8277,6937
1,confirmation,64189,35944,28602,20090,15497
2,total,91775,51348,40974,28367,22434


## 5. Outlier fences and encoding (fitted on exploration, applied to both)

Price is right-skewed, so its fences are set on log price (Q1 - 1.5 IQR, Q3 + 1.5 IQR). Property
types with fewer than 30 exploration rows merge into `Other`; the 100 most common exploration
amenities become 0/1 columns.

In [7]:
params = prep.fit_encoding(explore)
explore_c, caps_e = prep.apply_outlier_caps(explore, params)
confirm_c, caps_c = prep.apply_outlier_caps(confirm, params)
full_c = pd.concat([explore_c, confirm_c]).sort_values("id")
X = prep.encode(full_c, params)
size = {"rows": int(X.shape[0]), "encoded_columns": int(X.shape[1]),
        "cells": int(X.shape[0] * X.shape[1]),
        "numeric": len(prep.NUMERIC_FEATURES), "flags": len(prep.FLAG_FEATURES),
        "borough_dummies": len(params["boroughs"]), "room_dummies": len(params["room_types"]),
        "property_dummies": len(params["property_levels"]) + 1, "amenity_columns": len(params["amenities"]),
        "price_fence_low": params["price_low"], "price_fence_high": params["price_high"],
        "price_below_fence": caps_e["below_fence"] + caps_c["below_fence"],
        "price_above_fence": caps_e["above_fence"] + caps_c["above_fence"],
        "property_levels_kept": len(params["property_levels"]),
        "property_types_raw": int(raw.property_type.nunique())}
save_json(size, "size_check.json")
save_json({k: v for k, v in params.items()}, "encoding_params.json")
print(f"size check: {size['rows']:,} x {size['encoded_columns']} = {size['cells']:,} >= 3,000,000 -> {size['cells'] >= 3_000_000}")
print(f"price fences (GBP/night): {params['price_low']:.0f} to {params['price_high']:.0f}; "
      f"capped below {size['price_below_fence']:,}, above {size['price_above_fence']:,}")
print("encoded blocks:", {k: size[k] for k in ['numeric', 'flags', 'borough_dummies', 'room_dummies', 'property_dummies', 'amenity_columns']})

size check: 91,775 x 219 = 20,098,725 >= 3,000,000 -> True
price fences (GBP/night): 19 to 1366; capped below 26, above 514
encoded blocks: {'numeric': 48, 'flags': 12, 'borough_dummies': 33, 'room_dummies': 4, 'property_dummies': 22, 'amenity_columns': 100}


In [8]:
# Before-and-after table for Section 3 of the report
p_before = prep.parse_price(raw["price"])
before_after = pd.DataFrame([
    ("Rows", len(raw), len(full_c)),
    ("Attributes", raw.shape[1], X.shape[1]),
    ("Instances x attributes", raw.shape[0] * raw.shape[1], X.shape[0] * X.shape[1]),
    ("Columns with no data", audit["n_empty_columns"], 0),
    ("Duplicate units", int(log.loc[log.step == "same unit listed twice", "touched"].iloc[0]), 0),
    ("Price as text", int(p_before.notna().sum()), 0),
    ("Prices outside log-IQR fences", int(((p_before < params["price_low"]) | (p_before > params["price_high"])).sum()), 0),
    ("Minimum nights > 365", audit["min_nights_over_365"], 0),
    ("Review averages below 1", int((raw[prep.REVIEW_SCORE_COLS] < 1).any(axis=1).sum()), 0),
    ("Missing price (kept, flagged)", audit["price_missing"], int(full_c.price_gbp.isna().sum())),
    ("Missing rating (kept, flagged)", audit["rating_missing"], int(full_c.review_scores_rating.isna().sum())),
], columns=["measure", "before", "after"])
before_after.to_csv(TABLES / "before_after.csv", index=False)
before_after

,measure,before,after
0,Rows,91954,91775
1,Attributes,90,219
2,Instances x attributes,8275860,20098725
3,Columns with no data,15,0
4,Duplicate units,179,0
5,Price as text,59905,0
6,Prices outside log-IQR fences,542,0
7,Minimum nights > 365,24,0
8,Review averages below 1,7,0
9,"Missing price (kept, flagged)",32049,32012


## 6. Exploration (exploration split only)

The population the tests will use is **established entire homes**: room type `Entire home/apt`
with at least 10 reviews (Airbnb's own Superhost rule needs 10 stays, and averages of a handful of
reviews are noisy).

In [9]:
def established(df):
    return df[(df.room_type == "Entire home/apt") & (df.number_of_reviews >= 10) & df.review_scores_rating.notna()].copy()

ex_est = established(explore_c)
eda = {"exploration_listings": len(explore_c), "exploration_hosts": int(explore_c.host_id.nunique()),
       "established_entire_homes": len(ex_est)}

# Host concentration
conc = (explore_c.groupby("host_class")
        .agg(hosts=("host_id", "nunique"), listings=("id", "size"))
        .reindex(["single (1)", "small (2-9)", "large (10+)"]))
conc["share_hosts"] = 100 * conc.hosts / conc.hosts.sum()
conc["share_listings"] = 100 * conc.listings / conc.listings.sum()
eda["concentration"] = conc.round(2).to_dict(orient="index")
eda["largest_portfolio_exploration"] = int(explore_c.host_london_listings.max())
figures.host_concentration(conc.rename(index={"single (1)": "1 listing", "small (2-9)": "2-9 listings", "large (10+)": "10+ listings"}), FIGS)
conc

1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


,hosts,listings,share_hosts,share_listings
host_class,,,,
single (1),12372,12372,80.316801,44.848836
small (2-9),2772,8277,17.995326,30.004350
large (10+),260,6937,1.687873,25.146814


In [10]:
# Room-type mix by host class: are large hosts mainly entire homes?
mix = pd.crosstab(explore_c.host_class, explore_c.room_type, normalize="index").mul(100).round(1)
eda["entire_home_share_by_class"] = mix["Entire home/apt"].to_dict()
mix

room_type,Entire home/apt,Hotel room,Private room,Shared room
host_class,,,,
large (10+),79.6,0.1,20.2,0.2
single (1),64.8,0.0,35.0,0.1
small (2-9),56.4,0.2,43.2,0.2


In [11]:
# Rating distribution, single vs large hosts (established entire homes)
summ = {}
for cls in ["single (1)", "small (2-9)", "large (10+)"]:
    r = ex_est.loc[ex_est.host_class == cls, "review_scores_rating"]
    summ[cls] = {"n": int(len(r)), "mean": r.mean(), "median": r.median(), "q1": r.quantile(.25),
                 "q3": r.quantile(.75), "share_below_4_8": float((r < 4.8).mean() * 100),
                 "skewness": float(stats.skew(r))}
all_r = ex_est.review_scores_rating
summ["all"] = {"n": int(len(all_r)), "mean": all_r.mean(), "median": all_r.median(), "q1": all_r.quantile(.25),
               "q3": all_r.quantile(.75), "share_below_4_8": float((all_r < 4.8).mean() * 100),
               "share_below_4": float((all_r < 4.0).mean() * 100), "skewness": float(stats.skew(all_r)),
               "sd": all_r.std()}
eda["rating_summary"] = summ
figures.rating_ecdf(ex_est.loc[ex_est.host_class == "single (1)", "review_scores_rating"],
                    ex_est.loc[ex_est.host_class == "large (10+)", "review_scores_rating"], FIGS)
pd.DataFrame(summ).T.round(3)

1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


,n,mean,median,q1,q3,share_below_4_8,skewness,share_below_4,sd
single (1),3119.0,4.814,4.87,4.73,4.95,33.825,-1.668,NaN,NaN
small (2-9),2253.0,4.735,4.78,4.63,4.89,53.351,-1.265,NaN,NaN
large (10+),2425.0,4.639,4.68,4.50,4.82,70.928,-0.941,NaN,NaN
all,7797.0,4.737,4.79,4.62,4.91,51.007,-1.253,0.577,0.222


In [12]:
# Does the large-host share vary by borough? (a confounder if it does)
pl = ex_est[ex_est.host_class.isin(["single (1)", "large (10+)"])]
share = pl.groupby("neighbourhood_cleansed")["host_class"].apply(lambda s: 100 * (s == "large (10+)").mean())
figures.large_share_by_borough(share, FIGS)
eda["large_share_by_borough"] = {"min": float(share.min()), "min_borough": share.idxmin(),
                                 "max": float(share.max()), "max_borough": share.idxmax(),
                                 "median": float(share.median())}
# Rating also varies by borough, so borough must be held fixed:
br = pl.groupby("neighbourhood_cleansed")["review_scores_rating"].mean()
eda["rating_by_borough"] = {"min": float(br.min()), "min_borough": br.idxmin(), "max": float(br.max()), "max_borough": br.idxmax()}
# ... and by bedroom band
eda["large_share_by_bedrooms"] = pl.groupby("bedroom_band")["host_class"].apply(lambda s: round(100 * (s == "large (10+)").mean(), 1)).to_dict()
eda["rating_by_bedrooms"] = pl.groupby("bedroom_band")["review_scores_rating"].mean().round(3).to_dict()
print(eda["large_share_by_borough"], eda["rating_by_borough"], eda["large_share_by_bedrooms"], eda["rating_by_bedrooms"], sep="\n")

1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


{'min': 5.0, 'min_borough': 'Bexley', 'max': 71.7948717948718, 'max_borough': 'City of London', 'median': 28.57142857142857}
{'min': 4.596111111111111, 'min_borough': 'Hillingdon', 'max': 4.865074626865671, 'max_borough': 'Richmond upon Thames'}
{'0-1': 44.9, '2': 41.8, '3+': 44.0}
{'0-1': 4.717, '2': 4.754, '3+': 4.77}


In [13]:
# Hype words in descriptions (established entire homes with a description)
hx = ex_est[ex_est.description_words > 0].copy()
dist = hx.hype_count.value_counts().sort_index()
share_at_least = {k: float((hx.hype_count >= k).mean() * 100) for k in range(0, 8)}
eda["hype"] = {"n": len(hx), "share_zero": float((hx.hype_count == 0).mean() * 100),
               "share_at_least": share_at_least, "median_words": float(hx.description_words.median()),
               "top_words": pd.Series([w.lower() for t in hx.description for w in prep._HYPE_RE.findall(t)]).value_counts().head(8).to_dict()}
# Rule fixed before looking at accuracy: hyped = the largest k that still leaves >= 15% of listings
k_hyped = max(k for k, v in share_at_least.items() if v >= 15)
eda["hype"]["k_hyped"] = k_hyped
print("share with >= k hype words:", {k: round(v, 1) for k, v in share_at_least.items()})
print("hyped threshold k =", k_hyped, "| top words:", eda["hype"]["top_words"])

share with >= k hype words: {0: 100.0, 1: 57.6, 2: 23.8, 3: 8.4, 4: 3.0, 5: 0.9, 6: 0.3, 7: 0.1}
hyped threshold k = 2 | top words: {'perfect': 2336, 'beautiful': 934, 'stunning': 424, 'luxury': 400, 'perfectly': 355, 'unique': 332, 'beautifully': 329, 'iconic': 311}


In [14]:
# Accuracy by hype bucket (exploration only)
cap = 4
hx["bucket"] = pd.Categorical(np.where(hx.hype_count >= cap, f"{cap}+", hx.hype_count.astype(str)),
                              categories=[str(i) for i in range(cap)] + [f"{cap}+"], ordered=True)
figures.accuracy_by_hype(hx, FIGS)
acc = hx.groupby("bucket", observed=True)["review_scores_accuracy"].agg(["size", "mean", "median"])
eda["accuracy_by_hype"] = acc.round(4).to_dict(orient="index")
eda["accuracy_summary"] = {"median": float(hx.review_scores_accuracy.median()), "q1": float(hx.review_scores_accuracy.quantile(.25)),
                           "q3": float(hx.review_scores_accuracy.quantile(.75))}
acc

1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


,size,mean,median
bucket,,,
0,3295,4.774586,4.83
1,2629,4.777741,4.82
2,1202,4.776839,4.83
3,421,4.799074,4.85
4+,231,4.787662,4.82


In [15]:
# Obvious questions anyone would ask first (answered quickly on exploration data, for context)
ex_priced = explore_c[explore_c.price_gbp.notna()]
obvious = {
    "median price by room type": ex_priced.groupby("room_type").price_gbp_capped.median().round(0).to_dict(),
    "priciest boroughs (median entire home)": ex_priced[ex_priced.room_type == "Entire home/apt"].groupby("neighbourhood_cleansed").price_gbp_capped.median().sort_values().tail(3).round(0).to_dict(),
    "mean rating superhost vs not": ex_est.groupby("host_is_superhost").review_scores_rating.mean().round(3).to_dict(),
    "spearman rating vs number_of_reviews": float(stats.spearmanr(ex_est.review_scores_rating, ex_est.number_of_reviews).statistic),
}
eda["obvious"] = obvious
save_json(eda, "eda_summary.json")
obvious

{'median price by room type': {'Entire home/apt': 224.0,
  'Hotel room': 165.0,
  'Private room': 80.0,
  'Shared room': 33.0},
 'priciest boroughs (median entire home)': {'Kensington and Chelsea': 295.0,
  'City of London': 296.0,
  'Westminster': 304.0},
 'mean rating superhost vs not': {np.False_: 4.684, np.True_: 4.857},
 'spearman rating vs number_of_reviews': 0.01309811140813918}

## 7. Problem statements and pre-registered hypotheses

* **P1 (primary).** Among comparable entire homes (same borough and bedroom band), do listings run
  by large hosts (10+ London listings) get different guest ratings from listings run by
  single-listing hosts?
* **P2.** Among established entire homes, do listings whose descriptions pile on marketing
  superlatives score lower on *accuracy* (guests' "did it match the page?" rating) than plainly
  described ones?

All hypotheses, significance levels, sides, tests and smallest differences that matter were
written to `docs/preregistration.json` after the exploration above and **before** any test touched
the confirmation split. The cell below prints its SHA-256 so the file cannot change silently.

In [16]:
PREREG = ROOT / "docs" / "preregistration.json"
prereg = json.loads(PREREG.read_text())
print("preregistration sha256:", prep.sha256_of(PREREG))
ALPHA = prereg["family_alpha"]
SEED = prereg["seed"]
assert prereg["hype"]["k_hyped"] == eda["hype"]["k_hyped"]
pd.DataFrame(prereg["tests"]).T[["problem", "H0", "H1", "alpha", "side", "test", "smallest_difference"]]

preregistration sha256: 24da1c62de4068c1e93eb4ae3cc1d2e21bec14cb5eed2e4b5119655fecd42321


,problem,H0,H1,alpha,side,test,smallest_difference
T1,P1,beta = 0,beta != 0,0.05,two-sided,OLS with one intercept per stratum and host-clustered (CR1) standard errors; t with G - 1 df,0.10 star
T2,P1,theta = 1,theta != 1,0.05,two-sided,"Cochran-Mantel-Haenszel chi-square (1 df), Mantel-Haenszel pooled odds ratio",odds ratio outside 0.80-1.25
T3,P1,mu_L - mu_S = 0,mu_L - mu_S != 0,0.05,two-sided,Welch two-sample t-test,0.10 star
T4,P2,delta = 0,delta != 0,0.05,two-sided,"stratified permutation test (labels permuted within strata, 10,000 permutations); 95% CI by stratified bootstrap",0.10 star


## 8. Confirmatory tests (confirmation split only)

Samples: established entire homes in the confirmation split. P1 compares hosts with exactly one
London listing (S) with hosts holding 10 or more (L); the 2-9 group is not used. Strata are
borough x bedroom band (0-1, 2, 3+).

In [17]:
cf_est = established(confirm_c)
p1 = cf_est[cf_est.host_class.isin(["single (1)", "large (10+)"])].copy()
p1["large"] = p1.host_class.eq("large (10+)")
p1 = p1[p1.bedroom_band.notna()]
p1["stratum"] = p1.neighbourhood_cleansed + " | " + p1.bedroom_band
p1["below_bar"] = p1.review_scores_rating < prereg["tests"]["T2"]["bar"]
samples = {"p1_listings": len(p1), "p1_large": int(p1.large.sum()), "p1_single": int((~p1.large).sum()),
           "p1_large_hosts": int(p1.loc[p1.large, "host_id"].nunique()), "p1_hosts": int(p1.host_id.nunique()),
           "p1_strata": int(p1.stratum.nunique())}
samples

{'p1_listings': 12485,
 'p1_large': 5288,
 'p1_single': 7197,
 'p1_large_hosts': 477,
 'p1_hosts': 7674,
 'p1_strata': 99}

### Test 1 (P1): mean rating gap within strata, host-clustered standard errors

In [18]:
t1 = st.fe_cluster_diff(p1, "review_scores_rating", "large", "stratum", "host_id")
L = p1.loc[p1.large, "review_scores_rating"]; S = p1.loc[~p1.large, "review_scores_rating"]
t1_alt = {
    "welch_unadjusted": st.welch(L, S),
    "pooled_unadjusted": st.pooled_t(L, S),
    "brown_forsythe": st.brown_forsythe(L, S),
    "mann_whitney": st.mann_whitney(L, S),
    "stratified_permutation": st.stratified_permutation(p1.review_scores_rating, p1.large, p1.stratum,
                                                         "two-sided", n_perm=prereg["n_permutations"], seed=SEED),
}
t1["mean_large"], t1["mean_single"] = float(L.mean()), float(S.mean())
t1["verdict"] = st.practical_verdict(t1["ci_low"], t1["ci_high"], prereg["tests"]["T1"]["smallest_difference_value"])
print(json.dumps({k: (round(v, 6) if isinstance(v, float) else v) for k, v in t1.items()}, indent=1))
pd.DataFrame({k: {kk: vv for kk, vv in v.items() if not isinstance(vv, (list, dict, tuple))} for k, v in t1_alt.items()}).T

{
 "test": "stratum fixed effects + host-clustered SE",
 "estimate": -0.145021,
 "se": 0.01175,
 "statistic": -12.342425,
 "df": 7673,
 "p": 0.0,
 "ci_low": -0.168054,
 "ci_high": -0.121988,
 "clusters": 7674,
 "n": 12485,
 "strata": 99,
 "strata_with_both_groups": 96,
 "iid_se": 0.004094,
 "iid_statistic": -35.424769,
 "iid_df": 12385.0,
 "iid_p": 0.0,
 "mean_large": 4.661216,
 "mean_single": 4.814781,
 "verdict": "beyond the smallest difference that matters"
}


,test,n1,n2,mean1,mean2,estimate,se,statistic,df,p,ci_low,ci_high,sd1,sd2,prob_superiority,n_perm,strata_used,n,null_sd
welch_unadjusted,Welch t,5288,7197,4.661216,4.814781,-0.153565,0.003925,-39.128339,9511.628098,0.0,-0.161258,-0.145872,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pooled_unadjusted,pooled t,NaN,NaN,NaN,NaN,-0.153565,NaN,-40.732114,12483,0.0,-0.160955,-0.146175,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brown_forsythe,Brown-Forsythe,NaN,NaN,NaN,NaN,NaN,NaN,416.920061,NaN,0.0,NaN,NaN,0.238729,0.182452,NaN,NaN,NaN,NaN,NaN
mann_whitney,Mann-Whitney U,NaN,NaN,NaN,NaN,NaN,NaN,10985469.5,NaN,0.0,NaN,NaN,NaN,NaN,0.288653,NaN,NaN,NaN,NaN
stratified_permutation,stratified permutation,NaN,NaN,NaN,NaN,-0.148636,NaN,NaN,NaN,0.0001,NaN,NaN,NaN,NaN,NaN,10000,96,12443,0.004375


### Test 2 (P1): share below the 4.8 Superhost bar, Cochran-Mantel-Haenszel

In [19]:
t2 = st.cmh(p1, "below_bar", "large", "stratum")
t2_alt = st.pooled_2x2(t2["pooled_table"])
t2["share_large"], t2["share_single"] = t2_alt["share_treated"], t2_alt["share_control"]
t2["verdict"] = st.practical_verdict(t2["ci_low"], t2["ci_high"], prereg["tests"]["T2"]["smallest_difference_value"], scale="ratio")
# Sensitivity: one listing per host, so a big operator counts once
one = p1.sample(frac=1, random_state=SEED).drop_duplicates("host_id")
t2_one = st.cmh(one, "below_bar", "large", "stratum")
print(json.dumps({k: v for k, v in t2.items() if k != "pooled_table"}, indent=1, default=float))
print("pooled alternatives:", json.dumps(t2_alt, indent=1))
print("one-per-host sensitivity: OR", round(t2_one["estimate"], 3), "CI", round(t2_one["ci_low"], 3), round(t2_one["ci_high"], 3), "p", t2_one["p"])

{
 "test": "Cochran-Mantel-Haenszel",
 "statistic": 838.2062565714218,
 "df": 1,
 "p": 0.0,
 "estimate": 3.219950994962795,
 "ci_low": 2.9678779181861943,
 "ci_high": 3.4934335898487046,
 "strata_used": 96,
 "breslow_day_stat": 223.3607919281726,
 "breslow_day_p": 2.413513833232628e-12,
 "n_in_strata": 12443,
 "share_large": 0.6552571860816944,
 "share_single": 0.34647099930118797,
 "verdict": "beyond the smallest difference that matters"
}
pooled alternatives: {
 "chi2": 1162.027956979185,
 "chi2_df": 1,
 "chi2_p": 1.0909229260596714e-254,
 "z": 34.08853116488278,
 "z_p": 1.090922926059785e-254,
 "fisher_or": 3.5852095617448776,
 "fisher_p": 2.4253799007973116e-258,
 "share_treated": 0.6552571860816944,
 "share_control": 0.34647099930118797
}
one-per-host sensitivity: OR 3.058 CI 2.504 3.734 p 0.0


### Test 3 (P1): host-level means, Welch t-test (each host counts once)

In [20]:
hostlvl = p1.groupby(["host_id", "large"], as_index=False)["review_scores_rating"].mean()
HL = hostlvl.loc[hostlvl.large, "review_scores_rating"]; HS = hostlvl.loc[~hostlvl.large, "review_scores_rating"]
t3 = st.welch(HL, HS)
t3_alt = {"pooled": st.pooled_t(HL, HS), "brown_forsythe": st.brown_forsythe(HL, HS),
          "mann_whitney": st.mann_whitney(HL, HS),
          "permutation": st.permutation_mean_diff(HL, HS, n_resamples=prereg["n_permutations"], seed=SEED)}
t3["verdict"] = st.practical_verdict(t3["ci_low"], t3["ci_high"], prereg["tests"]["T3"]["smallest_difference_value"])
print(json.dumps(t3, indent=1, default=float))
pd.DataFrame({k: {kk: vv for kk, vv in v.items() if not isinstance(vv, (list, dict, tuple))} for k, v in t3_alt.items()}).T

{
 "test": "Welch t",
 "n1": 477,
 "n2": 7197,
 "mean1": 4.650431728583962,
 "mean2": 4.814781158816173,
 "estimate": -0.16434943023221038,
 "se": 0.00920779798905775,
 "statistic": -17.848939608310037,
 "df": 532.4005970372785,
 "p": 3.3885247213661005e-56,
 "ci_low": -0.18243750262932443,
 "ci_high": -0.14626135783509633,
 "verdict": "beyond the smallest difference that matters"
}


,test,estimate,statistic,df,p,ci_low,ci_high,sd1,sd2,prob_superiority,n_resamples
pooled,pooled t,-0.164349,-18.964943,7672,0.0,-0.181337,-0.147362,NaN,NaN,NaN,NaN
brown_forsythe,Brown-Forsythe,NaN,9.58016,NaN,0.001974,NaN,NaN,0.195539,0.182452,NaN,NaN
mann_whitney,Mann-Whitney U,NaN,819774.5,NaN,0.0,NaN,NaN,NaN,NaN,0.238795,NaN
permutation,permutation (difference in means),NaN,-0.164349,NaN,0.0002,NaN,NaN,NaN,NaN,NaN,10000


### Test 4 (P2): hyped vs plain descriptions, accuracy score, stratified permutation

In [21]:
k = prereg["hype"]["k_hyped"]
p2 = cf_est[(cf_est.description_words > 0) & ((cf_est.hype_count == 0) | (cf_est.hype_count >= k))].copy()
p2 = p2.sample(frac=1, random_state=SEED).drop_duplicates("host_id")  # one random listing per host
p2["hyped"] = p2.hype_count >= k
p2["stratum"] = p2.neighbourhood_cleansed + " | " + p2.host_class
y, g, s = p2.review_scores_accuracy, p2.hyped, p2.stratum
t4 = st.stratified_permutation(y, g, s, "two-sided", n_perm=prereg["n_permutations"], seed=SEED)
t4["ci_low"], t4["ci_high"] = st.stratified_bootstrap_ci(y, g, s, n_boot=prereg["n_bootstrap"], seed=SEED)
t4["n_hyped"], t4["n_plain"] = int(g.sum()), int((~g).sum())
t4["mean_hyped"], t4["mean_plain"] = float(y[g].mean()), float(y[~g].mean())
t4["verdict"] = st.practical_verdict(t4["ci_low"], t4["ci_high"], prereg["tests"]["T4"]["smallest_difference_value"])
t4_alt = {"welch_unstratified": st.welch(y[g], y[~g]),
          "mann_whitney": st.mann_whitney(y[g], y[~g]),
          "permutation_unstratified": st.permutation_mean_diff(y[g], y[~g], n_resamples=prereg["n_permutations"], seed=SEED)}
dose = cf_est[cf_est.description_words > 0].sample(frac=1, random_state=SEED).drop_duplicates("host_id")
sp = stats.spearmanr(dose.hype_count, dose.review_scores_accuracy)
t4_alt["spearman_dose_response"] = {"test": "Spearman", "statistic": float(sp.statistic), "p": float(sp.pvalue), "n": len(dose)}
print(json.dumps(t4, indent=1, default=float))
pd.DataFrame({k_: {kk: vv for kk, vv in v.items() if not isinstance(vv, (list, dict, tuple))} for k_, v in t4_alt.items()}).T

{
 "test": "stratified permutation",
 "estimate": 0.008461484447504948,
 "p": 0.0443955604439556,
 "n_perm": 10000,
 "strata_used": 92,
 "n": 7367,
 "null_sd": 0.004221053213785851,
 "ci_low": 0.00038721825949908546,
 "ci_high": 0.016413616530785744,
 "n_hyped": 2470,
 "n_plain": 4908,
 "mean_hyped": 4.825408906882592,
 "mean_plain": 4.820719233903831,
 "verdict": "within the no-practical-difference band"
}


,test,n1,n2,mean1,mean2,estimate,se,statistic,df,p,ci_low,ci_high,prob_superiority,n_resamples,n
welch_unstratified,Welch t,2470,4908,4.825409,4.820719,0.00469,0.004335,1.081781,5030.050919,0.279402,-0.003809,0.013188,NaN,NaN,NaN
mann_whitney,Mann-Whitney U,NaN,NaN,NaN,NaN,NaN,NaN,6160615.5,NaN,0.249531,NaN,NaN,0.508186,NaN,NaN
permutation_unstratified,permutation (difference in means),NaN,NaN,NaN,NaN,NaN,NaN,0.00469,NaN,0.279572,NaN,NaN,NaN,10000,NaN
spearman_dose_response,Spearman,NaN,NaN,NaN,NaN,NaN,NaN,0.017039,NaN,0.080647,NaN,NaN,NaN,NaN,10513


### Robustness numbers quoted in the report

* T1's clustered t uses G - 1 = (all hosts - 1) degrees of freedom; because only the large hosts
  carry the treatment, a conservative alternative uses (large hosts - 1).
* Some p-values underflow to 0 in floating point, so their size is reported from the log tail.
* A permutation p-value is itself an estimate; its Monte Carlo standard error is sqrt(p(1-p)/B).

In [22]:
extras = {
    "confirmation_established_entire_homes": len(cf_est),
    "excluded_small_host_listings": int((cf_est.host_class == "small (2-9)").sum()),
    "excluded_missing_bedrooms": int(cf_est[cf_est.host_class.isin(["single (1)", "large (10+)"])].bedroom_band.isna().sum()),
    "t1_conservative_df": int(samples["p1_large_hosts"] - 1),
    "t1_conservative_log10_p": float((np.log(2) + stats.t.logsf(abs(t1["statistic"]), samples["p1_large_hosts"] - 1)) / np.log(10)),
    "t1_log10_p": float((np.log(2) + stats.t.logsf(abs(t1["statistic"]), t1["df"])) / np.log(10)),
    "t1_se_inflation": float(t1["se"] / t1["iid_se"]),
    "t2_log10_p": float(stats.chi2.logsf(t2["statistic"], 1) / np.log(10)),
    "t2_breslow_day_df": int(t2["strata_used"] - 1),
    "t2_pooled_log10_p": float(stats.chi2.logsf(t2_alt["chi2"], 1) / np.log(10)),
    "t2_one_per_host_log10_p": float(stats.chi2.logsf(t2_one["statistic"], 1) / np.log(10)),
    "t3_log10_p": float((np.log(2) + stats.t.logsf(abs(t3["statistic"]), t3["df"])) / np.log(10)),
    "t4_p_mc_se": float(np.sqrt(t4["p"] * (1 - t4["p"]) / t4["n_perm"])),
    "t4_hyped_share_large_hosts": float(p2.loc[p2.hyped, "host_class"].eq("large (10+)").mean() * 100),
    "t4_plain_share_large_hosts": float(p2.loc[~p2.hyped, "host_class"].eq("large (10+)").mean() * 100),
    "t4_hosts": int(p2.host_id.nunique()),
}
extras

{'confirmation_established_entire_homes': 17929,
 'excluded_small_host_listings': 5439,
 'excluded_missing_bedrooms': 5,
 't1_conservative_df': 476,
 't1_conservative_log10_p': -29.831330055019876,
 't1_log10_p': -33.94315060080951,
 't1_se_inflation': 2.8701627097077167,
 't2_log10_p': -183.5744279378137,
 't2_breslow_day_df': 95,
 't2_pooled_log10_p': -253.96220593132676,
 't2_one_per_host_log10_p': -29.630453737363695,
 't3_log10_p': -55.46998934156778,
 't4_p_mc_se': 0.002059723152679085,
 't4_hyped_share_large_hosts': 7.28744939271255,
 't4_plain_share_large_hosts': 5.6642216788916055,
 't4_hosts': 7378}

## 9. Family-wise correction (Holm) and saved results

In [23]:
raw_p = {"T1": t1["p"], "T2": t2["p"], "T3": t3["p"], "T4": t4["p"]}
adj = st.holm(raw_p)
summary = []
for name, res in [("T1", t1), ("T2", t2), ("T3", t3), ("T4", t4)]:
    summary.append({"test": name, "estimate": res["estimate"], "ci_low": res["ci_low"], "ci_high": res["ci_high"],
                    "statistic": res.get("statistic", np.nan), "df": res.get("df", np.nan),
                    "p_raw": raw_p[name], "p_holm": adj[name],
                    "decision": "reject H0" if adj[name] < ALPHA else "fail to reject H0", "practical": res["verdict"]})
summary = pd.DataFrame(summary)
summary.to_csv(TABLES / "test_summary.csv", index=False)
save_json({"samples": samples, "T1": t1, "T1_alt": t1_alt, "T2": {k_: v for k_, v in t2.items()}, "T2_alt": t2_alt,
           "T2_one_per_host": {k_: v for k_, v in t2_one.items() if k_ != "pooled_table"}, "T3": t3, "T3_alt": t3_alt,
           "T3_n": {"large_hosts": int(len(HL)), "single_hosts": int(len(HS))},
           "T4": t4, "T4_alt": t4_alt, "holm": adj, "alpha": ALPHA, "extras": extras}, "test_results.json")
summary

,test,estimate,ci_low,ci_high,statistic,df,p_raw,p_holm,decision,practical
0,T1,-0.145021,-0.168054,-0.121988,-12.342425,7673.000000,1.139854e-34,2.279709e-34,reject H0,beyond the smallest difference that matters
1,T2,3.219951,2.967878,3.493434,838.206257,1.000000,0.000000e+00,0.000000e+00,reject H0,beyond the smallest difference that matters
2,T3,-0.164349,-0.182438,-0.146261,-17.848940,532.400597,3.388525e-56,1.016557e-55,reject H0,beyond the smallest difference that matters
3,T4,0.008461,0.000387,0.016414,NaN,NaN,4.439556e-02,4.439556e-02,reject H0,within the no-practical-difference band


### Summary figure: every estimate against its smallest difference that matters

In [24]:
figures.test_summary(
    [("T1  large vs single, within strata", t1["estimate"], t1["ci_low"], t1["ci_high"]),
     ("T3  large vs single, one per host", t3["estimate"], t3["ci_low"], t3["ci_high"]),
     ("T4  hyped vs plain (accuracy)", t4["estimate"], t4["ci_low"], t4["ci_high"])],
    ("T2  odds of a rating below 4.8", t2["estimate"], t2["ci_low"], t2["ci_high"]), FIGS)

1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


[PosixPath('/Users/nishant/Desktop/sem 7/Datasci/cse558-midterm/outputs/figures/fig5_test_summary.pdf'),
 PosixPath('/Users/nishant/Desktop/sem 7/Datasci/cse558-midterm/outputs/figures/fig5_test_summary.png')]

## 10. Copy the figures next to the LaTeX report

`report/` is self-contained (it can be uploaded to Overleaf as is), so the figure PDFs it uses are
refreshed from `outputs/figures/` here.

In [25]:
import shutil
for pdf in sorted(FIGS.glob("fig*.pdf")):
    shutil.copy(pdf, ROOT / "report" / pdf.name)
    print("copied", pdf.name)

copied fig1_rating_ecdf.pdf
copied fig2_host_concentration.pdf
copied fig3_large_share_by_borough.pdf
copied fig4_accuracy_by_hype.pdf
copied fig5_test_summary.pdf
